# Lab 1 — Build the Murshid skeleton

*Day 1, hour 3 · 50 minutes · pairs*

::: {.callout-note appearance="simple"}
**Objective** — walk the layered project, read the `LLMClient` protocol and the
adapter behind it, hold a windowed bilingual conversation from the CLI, and write
the context-budget table.

**Before you start** — `make doctor` all green, and the gateway running
(`make gateway`, or `docker compose up -d gateway`).

**You finish with** — `make chat` working in both languages, a window that
demonstrably forgets, and `docs/context_budget.md` with six justified lines.
:::

Every code cell below runs the same command the Makefile runs, through
`sys.executable` so it works with or without `make`. Run the setup cell first.

In [1]:
import os, pathlib, sys, re, subprocess, urllib.request, json

# pytest and ruff colour their output; those escapes render as noise once the
# notebook is published, so they come off here rather than per command.
ANSI = re.compile(chr(27) + "\[[0-9;]*m")

for cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (cand / "src" / "murshid").is_dir():
        os.chdir(cand); break
    if (cand / "murshid" / "src" / "murshid").is_dir():
        os.chdir(cand / "murshid"); break

sys.path.insert(0, "src")
os.environ["PYTHONUTF8"] = "1"
os.environ.setdefault("PYTHONPATH", "src")

def run(*args, quiet_logs=True):
    """Run a course command and print what it printed.

    quiet_logs drops the structured log lines so the boxed summary is readable;
    pass quiet_logs=False when the log IS the lesson.
    """
    out = subprocess.run([sys.executable, *args], capture_output=True, text=True,
                         encoding="utf-8", errors="replace")
    text = ANSI.sub("", out.stdout + out.stderr)
    if quiet_logs:
        text = "\n".join(l for l in text.splitlines()
                          if not l.startswith("20") or "[" not in l[:40])
    print(text.strip())
    return out.returncode

# The gateway is 127.0.0.1 on a laptop and `gateway` inside compose, so take it
# from the same environment variable the application routes through rather than
# hardcoding a host that is only right in one of the two places.
GATEWAY = os.environ.get("MURSHID_PRIMARY_BASE_URL", "http://127.0.0.1:8080/v1")
GATEWAY = GATEWAY.rsplit("/v1", 1)[0].rstrip("/")

def fault(payload):
    """Fault injection on the course gateway: the 429 storm and the outage drill."""
    req = urllib.request.Request(
        GATEWAY + "/admin/fault", method="POST",
        data=json.dumps(payload).encode(), headers={"content-type": "application/json"})
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.load(r)

def gateway_stats():
    with urllib.request.urlopen(GATEWAY + "/admin/stats", timeout=5) as r:
        return json.load(r)

try:
    with urllib.request.urlopen(GATEWAY + "/healthz", timeout=3) as r:
        print("gateway:", json.load(r)["models"])
except Exception:
    print(f"gateway at {GATEWAY} is NOT answering — start it first:")
    print("   make gateway      (or)   docker compose up -d gateway")
print("cwd:", pathlib.Path.cwd())

gateway: ['course-flagship', 'course-small', 'course-anthropic', 'murshid-onprem']
cwd: /srv


<>:5: SyntaxWarning: invalid escape sequence '\['
<>:5: SyntaxWarning: invalid escape sequence '\['
/tmp/ipykernel_7/1500538845.py:5: SyntaxWarning: invalid escape sequence '\['
  ANSI = re.compile(chr(27) + "\[[0-9;]*m")


## 0 · Five minutes with a demo that works on stage

`demo_v0.py` is sixty lines: hardcoded model name, prompt inline, no timeout, no
state, `print` streaming. It works.

Read it first and list **three defects it ships with** before running it.

In [2]:
print(pathlib.Path("demo_v0.py").read_text(encoding="utf-8")[:1200])

"""demo_v0 — the chatbot that works on stage.

    python demo_v0.py "How do I renew my commercial licence?"

Sixty lines, one afternoon, and it demonstrates beautifully. It is also the
starting point of Lab 1, where the exercise is to list what breaks in production
*before* anybody names the failure modes formally. A room full of engineers finds
most of them unprompted, which is why the list is not printed here.

Six defects are marked `# SMELL`. There are more than six. Do not read the markers
until you have made your own list — the point of the exercise is the list you write,
not the one you agree with.

Nothing in this file is imported by the application. It exists to be deleted, and
the commit that deletes it is the first commit of the course.
"""

import os
import sys

from openai import OpenAI  # SMELL 1: the application imports the provider SDK

client = OpenAI(
    base_url=os.environ.get("OPENAI_BASE_URL", "http://127.0.0.1:8080/v1"),
    api_key=os.environ.get("OPENAI_API_KE

Now run it. The question is the one the demo was built to answer.

In [3]:
run("demo_v0.py", "How do I renew my commercial licence?")

Traceback (most recent call last):
  File "/opt/venv/lib/python3.12/site-packages/httpx2/_transports/default.py", line 98, in map_httpcore_exceptions
    yield
  File "/opt/venv/lib/python3.12/site-packages/httpx2/_transports/default.py", line 245, in handle_request
    resp = self._pool.handle_request(req)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/venv/lib/python3.12/site-packages/httpcore2/_sync/connection_pool.py", line 242, in handle_request
    raise exc from None
  File "/opt/venv/lib/python3.12/site-packages/httpcore2/_sync/connection_pool.py", line 224, in handle_request
    response = connection.handle_request(pool_request.request)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/venv/lib/python3.12/site-packages/httpcore2/_sync/connection.py", line 94, in handle_request
    raise exc
  File "/opt/venv/lib/python3.12/site-packages/httpcore2/_sync/connection.py", line 74, in handle_request
    stream = self._connect(request)
             

1

::: {.callout-important}
## The directory says SAR 200

The demo invented a fee, fluently, for the exact question it was built to demo —
because its service facts live in a string literal that nothing checks, and it has
no rule about what to do when it does not know.

Nobody notices unless they already know the right answer. That is the failure mode
this entire course is arranged around, and it is on screen in the first ten minutes
of day one.
:::

## 1 · The skeleton (10 min)

The layout is not decoration: it is what makes the CI check in
`tests/test_architecture.py` meaningful. That test walks the AST of everything
under `src/murshid/` and fails if any module outside `llm/` imports `openai` or
`anthropic`.

In [4]:
run("-m", "pytest", "tests/test_architecture.py", "-q")

......                                                                   [100%]


0

## 2 · The boundary (15 min)

Everything the application is allowed to know about a provider is in
`src/murshid/llm/interfaces.py`. Read the protocol itself — four lines.

In [5]:
import inspect
from murshid.llm.interfaces import LLMClient, LLMRequest, LLMResponse
print(inspect.getsource(LLMClient))

@runtime_checkable
class LLMClient(Protocol):
    """Everything the application is allowed to know about a model provider."""

    def complete(self, request: LLMRequest) -> LLMResponse: ...

    def stream(self, request: LLMRequest) -> Iterator[StreamChunk]: ...



`OpenAICompatClient.complete()` does four things, each of which is a rule from the
theory rather than a detail of the SDK:

1. it resolves `request.model_alias` **through the route config** — never a literal
   model id;
2. it sets `max_tokens`, always;
3. it sets `max_retries=0` on the SDK client, because *we* own retry policy;
4. it returns the **concrete** `model_id` that answered, plus `usage`, plus latency.

Watch all four arrive in one line of output.

In [6]:
run("-m", "murshid.cli", "ask", "How do I renew my commercial licence?")

[faq → course-flagship via primary] 632ms, 1413 in (1378 cached) / 138 out, 0.971 halalas
About Renewing a commercial registration (CR):
- Fee: SAR 200 for each year of renewal
- Processing time: Same working day once payment clears
- Documents required: Valid national ID or Iqama; Current municipality licence; Zakat certificate for the last closed year
- Steps:
  1. Sign in to the portal with your national ID
  2. Open Business Services and choose Renew Commercial Registration
  3. Confirm the activity and the renewal period
  4. Pay through SADAD and download the renewed certificate
If you need more help you can contact Any Digital Government Services Authority centre, or the 24/7 line 199.
{"route": "primary+fallback", "faq_alias": "murshid-default", "service_alias": "murshid-default", "routing_enabled": false, "cascade": false, "cache": false, "semantic_cache": false, "faq_prompt": "answer_faq.v5", "event": "assistant_built", "level": "info", "timestamp": "2026-09-06T11:29:44.46495

0

SAR 200 this time, from the directory, because the answer is composed from facts
that arrived in the prompt.

Ask the identical question again and watch `cached` climb — that is Module 6
arriving early, and it is the reason `cache_prefix_messages` exists on a Module 1
type.

In [7]:
run("-m", "murshid.cli", "ask", "How do I renew my commercial licence?")

[faq → course-flagship via primary] 449ms, 1413 in (1378 cached) / 138 out, 0.971 halalas
About Renewing a commercial registration (CR):
- Fee: SAR 200 for each year of renewal
- Processing time: Same working day once payment clears
- Documents required: Valid national ID or Iqama; Current municipality licence; Zakat certificate for the last closed year
- Steps:
  1. Sign in to the portal with your national ID
  2. Open Business Services and choose Renew Commercial Registration
  3. Confirm the activity and the renewal period
  4. Pay through SADAD and download the renewed certificate
If you need more help you can contact Any Digital Government Services Authority centre, or the 24/7 line 199.
{"route": "primary+fallback", "faq_alias": "murshid-default", "service_alias": "murshid-default", "routing_enabled": false, "cascade": false, "cache": false, "semantic_cache": false, "faq_prompt": "answer_faq.v5", "event": "assistant_built", "level": "info", "timestamp": "2026-09-06T11:29:46.93391

0

## 3 · Windowed state and the CLI (10 min)

**LLM APIs are stateless.** Every request carries the whole conversation. "Memory"
is an application concern, and `ConversationState` ships with `max_turns=8`.

`make chat` is interactive, so here is the same thing without a terminal: nine
exchanges, then a question about the first one.

In [8]:
from murshid.domain.session import ConversationState

state = ConversationState(max_turns=8)
for i in range(9):
    state.add_user(f"question {i}")
    state.add_assistant(f"answer {i}")

msgs = state.messages()
print(f"window: {len(msgs)} messages, max {state.max_turns} turns")
print("oldest kept:", msgs[0].content)
print("newest kept:", msgs[-1].content)
print("\nis 'question 0' still in the window?", any("question 0" == m.content for m in msgs))

window: 16 messages, max 8 turns
oldest kept: question 1
newest kept: answer 8

is 'question 0' still in the window? False


::: {.callout-important}
## The moment statelessness lands — do not skip it

The window forgot turn 1, on cue. There is nothing wrong. A window is a *decision*
with a cost curve, not an implementation detail — and the alternative, unbounded
history, is a bill that grows every turn until the context overflows for your most
engaged users first.

Pinned by `tests/domain/test_ticket_and_session.py::test_the_window_forgets_the_oldest_turn`.
:::

## 4 · The context budget (10 min)

Write `docs/context_budget.md`: allocate a 16k-token request budget across the
system prompt, the service directory, tool schemas, windowed history, this turn,
and output — and **justify each line**.

Measure rather than guess.

In [9]:
from murshid.domain.directory import rendered_directory
from murshid.llm.tokens import count

for lang in ("en", "ar"):
    print(f"{lang}: {count(rendered_directory(lang))} tokens")

en: 1127 tokens
ar: 1352 tokens


Two questions your document has to answer:

- At ~600 tokens per turn, which turn overflows a 16k window with *unbounded*
  history? Show the arithmetic.
- Which line of your budget is the largest, and is it in the cacheable prefix or the
  volatile tail? Module 6 will make you care.

The reference version is in `docs/context_budget.md`. Compare after you have
written yours, not before.

## 5 · Reliability, under a real fault

The gateway can inject a real outage. This is the drill, end to end: turn it on,
ask a question, watch the retry and the fallback, turn it off.

In [10]:
print(fault({"mode": "overload", "seconds": 60, "model": "course-flagship"}))

{'fault': {'mode': 'overload', 'until': 1788694248.280382, 'model': 'course-flagship'}}


Now ask. Keep the log this time — the retry and the failover *are* the lesson.

In [11]:
run("-m", "murshid.cli", "ask", "How do I renew my commercial licence?", quiet_logs=False)

[faq → murshid-onprem via vllm] 2860ms, 1413 in (0 cached) / 138 out, 0.282 halalas
About Renewing a commercial registration (CR):
- Fee: SAR 200 for each year of renewal
- Processing time: Same working day once payment clears
- Documents required: Valid national ID or Iqama; Current municipality licence; Zakat certificate for the last closed year
- Steps:
  1. Sign in to the portal with your national ID
  2. Open Business Services and choose Renew Commercial Registration
  3. Confirm the activity and the renewal period
  4. Pay through SADAD and download the renewed certificate
If you need more help you can contact Any Digital Government Services Authority centre, or the 24/7 line 199.
{"route": "primary+fallback", "faq_alias": "murshid-default", "service_alias": "murshid-default", "routing_enabled": false, "cascade": false, "cache": false, "semantic_cache": false, "faq_prompt": "answer_faq.v5", "event": "assistant_built", "level": "info", "timestamp": "2026-09-06T11:29:49.784440Z"}
{

0

Turn the fault off before moving on.

In [12]:
print(fault({"mode": "off"}))

{'fault': {'mode': 'off'}}


Three things to notice in that log:

1. the retry **honoured the header** rather than guessing a backoff;
2. attempts were **capped** — retries multiply cost and tail latency;
3. the citizen got an answer, from the on-premise route, and never knew.

## 6 · Commit (5 min)

```bash
git add -A
git commit -m "feat: murshid skeleton with provider boundary and windowed state"
```

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `✗ route primary` in `make doctor` | the gateway is not running | `make gateway` in another terminal |
| `AuthenticationError` against a real gateway | `.env` not loaded, or the wrong variable name | `cp configs/settings.example.env .env`; the prefix is `MURSHID_` |
| Responses ignore earlier turns | state not replayed into `messages` | history serialises system first, then turns oldest → newest |
| Arabic renders as boxes | console code page | Windows Terminal or VS Code, and `PYTHONUTF8=1` |
| `finish_reason="length"` mid-sentence | `max_tokens` too low | raise it per your budget — and note that truncation is *silent* |
| Everything is slow (~2 s per call) | `localhost` resolving to IPv6 first | the config uses `127.0.0.1` for exactly this reason |

## If you finish early

Stream the answer instead, and note that TTFT and total are reported separately.
That distinction is tomorrow's warm-up and Module 6's headline.

In [13]:
run("-m", "murshid.cli", "stream", "What documents do I need to renew my commercial registration?")

About Renewing a commercial registration (CR):
- Fee: SAR 200 for each year of renewal
- Processing time: Same working day once payment clears
- Documents required: Valid national ID or Iqama; Current municipality licence; Zakat certificate for the last closed year
- Steps:
  1. Sign in to the portal with your national ID
  2. Open Business Services and choose Renew Commercial Registration
  3. Confirm the activity and the renewal period
  4. Pay through SADAD and download the renewed certificate
If you need more help you can contact Any Digital Government Services Authority centre, or the 24/7 line 199.

[TTFT 377ms · total 451ms · 1406 in (1378 cached) / 138 out]


0